In [ ]:
import os
import pandas as pd
import numpy as np

# 1. 파일 경로 설정
DATA_DIR = "dataset"
crops_path = os.path.join(DATA_DIR, "exotic_crops_100_environment_and_economy.csv")
climate_path = os.path.join(DATA_DIR, "korea10year.csv")

# 2. 데이터 로드
df_crops = pd.read_csv(crops_path)

try:
    df_climate_raw = pd.read_csv(climate_path, encoding='utf-8-sig')
except:
    df_climate_raw = pd.read_csv(climate_path, encoding='cp949')

# 3. 컬럼명 정제
df_climate_raw.columns = df_climate_raw.columns.str.strip()

# 기상청 컬럼 자동 매핑
min_temp_col = [c for c in df_climate_raw.columns if '최저기온' in c][0]
max_temp_col = [c for c in df_climate_raw.columns if '최고기온' in c][0]
avg_temp_col = [c for c in df_climate_raw.columns if '평균기온' in c][0]
rhm_col = [c for c in df_climate_raw.columns if '상대습도' in c][0]
rn_col = [c for c in df_climate_raw.columns if '강수량' in c][0]

# 4. 10년 기후 통계 집계
df_climate = df_climate_raw.groupby('지점명').agg({
    min_temp_col: 'min',       # 10년 극최저기온
    max_temp_col: 'max',       # 10년 극최고기온
    avg_temp_col: 'mean',      # 10년 연평균 기온
    rhm_col: 'mean',           # 10년 평균 습도
    rn_col: lambda x: x.sum() / 10  # 연평균 강수량
}).reset_index()

df_climate.columns = ['region', 'min_temp', 'max_temp', 'avg_temp', 'avg_rhm', 'annual_rn']

# 5. Cross Join (지역 x 작물)
df_climate['key'] = 1
df_crops['key'] = 1
df_merged = pd.merge(df_climate, df_crops, on='key').drop('key', axis=1)

# 컬럼명 표준화
rename_map = {
    '작물명': 'crop_name',
    '생육적온_최저(℃)': 'opt_temp_min',
    '생육적온_최고(℃)': 'opt_temp_max',
    '한계생육온도(℃)': 'frost_limit_temp',
    '적정습도(%)': 'opt_humidity',
    '토양pH_최저': 'soil_ph_min',
    '토양pH_최고': 'soil_ph_max',
    '수익성(1-5)': 'profit_score'
}
df_merged = df_merged.rename(columns=rename_map)

# 6. 고도화된 Feature Engineering (파생 변수 생성)
# (1) 작물의 최적 평균 생육 온도
df_merged['opt_temp_avg'] = (df_merged['opt_temp_min'] + df_merged['opt_temp_max']) / 2.0

# (2) 기후-작물 격차 피처
df_merged['temp_diff_from_opt'] = np.abs(df_merged['avg_temp'] - df_merged['opt_temp_avg'])
df_merged['frost_safety_margin'] = df_merged['min_temp'] - df_merged['frost_limit_temp']

# 7. 연속형 Target Score (0~100점 점수화) 생성
# - 온도 격차가 적을수록, 한파 마진이 안전할수록 높은 점수 부여
def calculate_continuous_score(row):
    # 온도 적합 점수 (지수 감소 함수)
    t_score = np.exp(-0.08 * (row['temp_diff_from_opt'] ** 2)) * 100
    
    # 한파 안전 점수
    f_margin = row['frost_safety_margin']
    if f_margin >= 5:
        f_score = 100
    elif f_margin >= 0:
        f_score = 60 + (f_margin / 5.0) * 40
    else:
        f_score = max(0, 60 + f_margin * 12)  # 한계온도 미만시 감점 폭 증가
        
    return round(float((t_score * 0.6) + (f_score * 0.4)), 2)

df_merged['suitability_score'] = df_merged.apply(calculate_continuous_score, axis=1)

# 다중 등급 Target (0: 불가능, 1: 주의, 2: 적합, 3: 최적)
df_merged['suitability_grade'] = pd.cut(
    df_merged['suitability_score'], 
    bins=[-1, 40, 65, 85, 100], 
    labels=[0, 1, 2, 3]
).astype(int)

# 기존 이진 타깃(0 또는 1)도 유지
df_merged['suitability'] = (df_merged['suitability_score'] >= 65).astype(int)

# 8. 저장
output_path = os.path.join(DATA_DIR, "processed_ml_dataset.csv")
df_merged.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"✅ 고도화 전처리 완료! 저장 위치: {output_path} (총 {len(df_merged)}행)")